In [1]:
import os, json, pathlib, hashlib
import boto3, joblib, sklearn
from botocore.config import Config
from botocore.exceptions import ClientError
from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression

root = pathlib.Path('/opt/app-root/src')
marker = root / 'box1-persistence.txt'
marker.write_text('box1-ai-test persistent storage verified\n')
s3 = boto3.client('s3', endpoint_url=os.environ['AWS_S3_ENDPOINT'], region_name=os.environ['AWS_DEFAULT_REGION'], config=Config(signature_version='s3v4', s3={'addressing_style': 'path'}))
bucket = os.environ['AWS_S3_BUCKET']
payload = b'Box1 OpenShift AI notebook S3 round trip'
s3.put_object(Bucket=bucket, Key='validation/roundtrip.txt', Body=payload)
assert s3.get_object(Bucket=bucket, Key='validation/roundtrip.txt')['Body'].read() == payload
print('S3 round trip: PASS; TLS verification enabled')
try:
    s3.head_bucket(Bucket='first.bucket')
    raise AssertionError('Unexpected access to unrelated bucket')
except ClientError as error:
    assert error.response['ResponseMetadata']['HTTPStatusCode'] == 403, error.response['Error']['Code']
    print('Unrelated bucket access: DENIED (403)')
X, y = load_iris(return_X_y=True)
model = LogisticRegression(max_iter=200).fit(X, y)
model_path = root / 'model.joblib'
joblib.dump(model, model_path)
s3.upload_file(str(model_path), bucket, 'models/iris/model.joblib')
settings = {'name':'iris', 'implementation':'mlserver_sklearn.SKLearnModel', 'parameters':{'uri':'/mnt/models/model.joblib', 'version':'1'}}
s3.put_object(Bucket=bucket, Key='models/iris/model-settings.json', Body=json.dumps(settings).encode())
assert model.predict([[5.1,3.5,1.4,0.2]]).tolist() == [0]
print('Notebook execution: PASS; Iris prediction = 0')
print('Model uploaded: models/iris/model.joblib; sklearn', sklearn.__version__)
print('Persistent marker:', marker.read_text().strip())

S3 round trip: PASS; TLS verification enabled
Unrelated bucket access: DENIED (403)
Notebook execution: PASS; Iris prediction = 0
Model uploaded: models/iris/model.joblib; sklearn 1.8.0
Persistent marker: box1-ai-test persistent storage verified


In [1]:
from pathlib import Path
import json, requests
root = Path('/opt/app-root/src')
assert (root/'box1-persistence.txt').read_text() == 'box1-ai-test persistent storage verified\n'
assert (root/'model.joblib').stat().st_size == 975
assert (root/'box1-ai-smoke.ipynb').exists()
print('Persistence after dashboard Stop / Start: PASS')
print('Saved notebook, marker, and model survived workbench pod recreation.')

Persistence after dashboard Stop / Start: PASS
Saved notebook, marker, and model survived workbench pod recreation.


In [2]:
import requests, json
request = {'inputs':[{'name':'input-0','shape':[1,4],'datatype':'FP64','data':[[5.1,3.5,1.4,0.2]]}]}
response = requests.post('http://iris-predictor.box1-ai-test.svc.cluster.local:8080/v2/models/iris/infer', json=request, timeout=30)
response.raise_for_status()
result = response.json()
assert result['outputs'][0]['data'] == [0], result
print('KServe model inference: PASS (HTTP', response.status_code, ')')
print(json.dumps(result, indent=2))

KServe model inference: PASS (HTTP 200 )
{
  "model_name": "iris",
  "model_version": "1",
  "id": "3e901a3f-47fd-4a84-912e-260bef2fea18",
  "parameters": {},
  "outputs": [
    {
      "name": "predict",
      "shape": [
        1,
        1
      ],
      "datatype": "INT64",
      "parameters": {
        "content_type": "np"
      },
      "data": [
        0
      ]
    }
  ]
}
